In [2]:
#import numpy as np
from jinja2 import Template
import os
from jinja2 import Environment, StrictUndefined
env = Environment(undefined=StrictUndefined)

In [3]:
def apply_with_fail(txt,**kwargs):
    txt_template = env.from_string(txt)
    for k in kwargs:
        assert '{{' + k in txt, f'{k} not in template!'
    
    return txt_template.render(kwargs)


# PreTrain template

In [5]:
s_txt = '''#!/bin/bash
#SBATCH --job-name={{exp_name_pt}}   # Job name
#SBATCH --time=120:00:00               # Time limit hrs:min:sec
#SBATCH --output=/mnt/ML/ModelsTrainResults/katya.ivantsiv/SLURM/job_id_%j_job_name_%x.txt   # Standard output and error log
#SBATCH --gres=gpu:{{n_gpus}}
#SBATCH --tasks={{n_gpus}}
#SBATCH --cpus-per-task=8
#SBATCH --mem={{160000*n_gpus}}
#SBATCH --partition={{partition}}
#SBATCH --nice={{nice}}
#SBATCH --qos={{qos}}
{{node_list}}

export OPENBLAS_NUM_THREADS=2
export GOTO_NUM_THREADS=2
export OMP_NUM_THREADS=2

source /home/katya.ivantsiv/python-envs/venv-fairseq/bin/activate
cd {{project_path}}
export PYTHONPATH="$PYTHONPATH:{{project_path}}/:{{project_path}}/examples/data2vec"
whereis python

'''

srun_txt = '''srun bash -c '\\
\\
\\python fairseq_cli/hydra_train.py -m --config-dir examples/data2vec/config/v2 \\
--config-name {{config_name}} \\
optimization.lr=[{{lr}}] \\
+task.data={{task_data}} \\
dataset.train_subset={{train_datasets}} \\
hydra.sweep.dir={{sweep_dir}} \\
distributed_training.distributed_world_size={{n_gpus}} \\
dataset.num_workers=8 \\
+task.min_sample_size=250 \\
+task.max_sample_size=2000 \\
+task.use_h5={{use_h5}} \\
optimization.max_update={{max_updates}} \\
common.fp16={{fp16}} \\
common.bf16={{bf16}} \\
dataset.max_tokens={{max_tokens}} \\
optimization.update_freq=[{{update_frequency}}] \\
+model.modalities.laser.laser_conv_input_dim={{num_dims}} \\
+model.modalities.laser.batch_norm_input={{bn}} \\
"model.modalities.laser.feature_encoder_spec=\\"{{encoder_spec}}\\"" \\
{{H_W}} \\
+model.modalities.laser.conv_2plus1d={{conv_2plus1d}} \\
+task.feature_description={{features_base_path|default()}} \\
"+task.feat_idx=\\"{{channels|default((0,1,2,3))}}\\"" \\
+task.pretraining=True \\
+model.modalities.laser.avg_pool={{avg_pool}} \\
+model.modalities.laser.encoder={{encoder}} \\
+model.modalities.laser.skip_connection={{skip_connection}} \\
+model.modalities.laser.timm_output_dim={{timm_output_dim}} \\
+model.modalities.laser.timm_time_strides={{timm_time_strides}} \\
+model.modalities.laser.timm_pretrained_model_name={{timm_pretrained_model_name}} \\
+model.modalities.laser.GenericEncoder_uuid={{GenericEncoder_uuid}} \\
+model.modalities.laser.res_encoder_output_dim={{res_encoder_output_dim}} \\
+model.modalities.laser.res_encoder_activation={{res_encoder_activation}} \\
+model.SDPA={{SDPA}} \\
checkpoint.keep_interval_updates=10 \\
checkpoint.keep_last_epochs=20 \\
dataset.disable_dry_run=True \\
+task.div_clip={{div_clip}} \\
+model.modalities.laser.SR100_support={{SR100_support}} \\
+clearml.logging=False \\
+clearml.project_name="Fairseq/burst/encoders" \\
+clearml.task_name='$task_name' \\
+clearml.continue_last_task=True \\
'
'''

pretrain_txt = s_txt + srun_txt

# Finetune template

In [16]:
s_txt = '''#!/bin/bash
#SBATCH --job-name={{exp_name_ft}}   # Job name
#SBATCH --time=120:00:00               # Time limit hrs:min:sec
#SBATCH --output=/mnt/ML/ModelsTrainResults/katya.ivantsiv/SLURM/job_id_%j_job_name_%x.txt   # Standard output and error log
#SBATCH --gres=gpu:{{n_gpus}}
#SBATCH --tasks={{n_gpus}}
#SBATCH --cpus-per-task=8
#SBATCH --mem={{160000*n_gpus}}
#SBATCH --partition={{partition}}
#SBATCH --nice={{nice}}
#SBATCH --qos={{qos}}
{{node_list}}

export OPENBLAS_NUM_THREADS=2
export GOTO_NUM_THREADS=2
export OMP_NUM_THREADS=2

source /home/katya.ivantsiv/python-envs/venv-fairseq/bin/activate
cd {{project_path}}
export PYTHONPATH="$PYTHONPATH:{{project_path}}/:{{project_path}}/examples/data2vec"
whereis python
'''
srun_txt = '''srun bash -c '\\
\\
python fairseq_cli/hydra_train.py -m --config-dir examples/wav2vec/config/finetuning \\
--config-name {{config_name}} \\
optimization.lr=[{{lr}}] \\
+task.data={{task_data}} \\
hydra.sweep.dir={{sweep_dir}} \\
distributed_training.distributed_world_size={{n_gpus}} \\
dataset.num_workers=10 \\
optimization.max_update={{max_updates}} \\
dataset.max_tokens={{max_tokens}} \\
dataset.max_tokens_valid=15000 \\
+criterion.units_ctc_loss=False \\
+model.hubert_pred={{hubert_soft}} \\
+model.soft_hubert={{hubert_soft}} \\
+model.hubert_asr={{hubert_asr}} \\
+model.extra_trans_blocks_depth={{extra_trans_blocks_depth}} \\
+model.use_choice_sentences={{use_choice_sentences}} \\
+model.lip_embds={{lip_embds}} \\
+task.lip_reading_path={{lip_reading_path}} \\
+taks.lip_reading_treshold={{lip_reading_treshold}} \\
+task.use_h5={{use_h5}} \\
model.mask_prob={{mask_prob}} \\
+criterion.calib_ce=False \\
optimization.update_freq=[{{update_frequency}}] \\
+model.w2v_path={{pretrained_path}} \\
common.fp16={{fp16}} \\
common.bf16={{bf16}} \\
model.feature_grad_mult={{feature_grad_mult}} \\
+task.feature_description={{features_base_path|default()}} \\
"+task.feat_idx=\\"{{channels|default((0,1,2,3))}}\\"" \\
dataset.valid_subset={{validation_datasets}} \\
dataset.train_subset={{train_datasets}} \\
+model.SDPA={{SDPA}} \\
checkpoint.keep_interval_updates=10 \\
checkpoint.keep_last_epochs=20 \\
dataset.disable_dry_run=True \\
+task.div_clip={{div_clip}} \\
+clearml.logging=False \\
+clearml.project_name="Fairseq/burst/encoders" \\
+clearml.task_name='$task_name' \\
+clearml.continue_last_task=True \\
'
'''

# encoder_search_finetune_v3
# one_sensor_finetune_v1

finetune_txt = s_txt + srun_txt

In [7]:
import hashlib
def hash_encoder(encoder_config):
    # Convert the encoder configuration to a string
    encoder_config_str = str(encoder_config)
    
    # Calculate the SHA-256 hash of the string
    sha256_hash = hashlib.sha256(encoder_config_str.encode()).hexdigest()
    
    # Take the first 8 characters of the hash as the encoder name
    encoder_name = sha256_hash[:8]
    
    # Store the encoder name as the key and the configuration as the value
    # encoder_mapping[encoder_name] = encoder_config
    with open(f'/home/katya.ivantsiv/slurm/d2v/encoders/{encoder_name}.txt','wt') as f:
        f.write(encoder_config_str)
    print(encoder_config, encoder_name)
    return encoder_name


def channels_to_name(channels):
    import numpy as np
    if channels == (0,1,2,3):
        return ''
    
    if np.all(np.sort(channels) == np.arange(min(channels), 1 + max(channels))):
        channels_txt = f'_{min(channels)}to{max(channels)}'
        #print(channels_txt)
        return channels_txt
    else:
        return '_'+'_'.join([str(i) for i in np.sort(channels)])
    
channels_to_name((5,6,7))

'_5to7'

## Experiments

In [18]:
import subprocess

# Which config to use, currently only those are supported
pretrain_config_name = 'base_laser_only_enc_search.yaml'
# pretrain_config_name =  'large_laser_only_task_conv3d_v1.yaml'
finetune_config_name = 'vox_10h_laser.yaml'

username = 'katya.ivantsiv'
project_path = '/home/katya.ivantsiv/q-fairseq' # branch: sensor-optimization-tests

dataset = '100fps' #['100fps', '50fps']
sample_mode = 'linear' #['linear', 'burst']
encoder = 'LaserEncoder_V2Conv2D' # [LaserEncoder_V2Conv2D, LaserEncoder_V2] # conv2D, conv3D

split_path = f'/mnt/ML/Development/ML_Data_DB/v2/splits/other/low_FPS_features_splits/{sample_mode}/{dataset}'
pretrain_task_data = split_path
finetune_task_data = split_path
train_datasets = 'train'
validation_datasets = 'valid_lip_general@valid_loud_general@valid_silent_general@valid_lip_tiny_story@valid_loud_tiny_story@valid_silent_tiny_story'

div_clip = 32 #o.f param

if dataset == '100fps' and sample_mode == 'linear':
    features_base_path = '/mnt/ML/Development/dolev.orgad/low_fps_Q_Features/v2_420x800_optical_flow_quarter_simd/linear_100fps/features'
elif dataset == '50fps' and sample_mode == 'linear':
    features_base_path = '/mnt/ML/Development/dolev.orgad/low_fps_Q_Features/v2_420x800_optical_flow_quarter_simd/linear_50fps/features'
elif dataset == '50fps' and sample_mode == 'burst':
    features_base_path = '/mnt/ML/Development/dolev.orgad/low_fps_Q_Features/v2_420x800_optical_flow_quarter_simd/burst_50fps/features'
elif dataset == '100fps' and sample_mode == 'burst':
    features_base_path = '/mnt/ML/Development/dolev.orgad/low_fps_Q_Features/v2_420x800_optical_flow_quarter_simd/burst_100fps/features'



partition = 'A6000_L40S_VAST'
nice = 0
qos = 'normal'
SR100_support = 'false'

n_gpus = 4
max_updates_pre = 150_000
max_updates_fine = 150_000 
max_tokens_pre = 7600 
max_tokens_fine = 40000 
update_freq_pre = 3
update_freq_fine = 3
lr_pre  = 0.0004
lr_fine  = 0.0002

########################################

fp16 = 'True'
bf16 = 'False'
SDPA = 'False' # make the model run Faster

extra_trans_blocks_depth = 1
use_choice_sentences = 0
lip_embds  =          'True'
lip_reading_path = '/mnt/ML/Production/ML_Processed_Data/Audio_Features/lip_embedding/vdssskip6th_retinaface_ts1_ss4/results_summary_250522_slim_lmdb'
lip_reading_treshold = 0.5

use_h5 =        'True'
mask_prob_fine = 0.05

hubert_soft  =        'True'
hubert_asr   =        'True'
avg_pool     =        'False'
conv_2plus1d =        'False'
bn =                  'True'
normalize    =        'False'
skip_connection =     'True'

GenericEncoder_params_str = ''
GenericEncoder_uuid = ''

res_encoder_activation = ''
res_encoder_output_dim = 0

timm_output_dim = 0
timm_time_strides = 0
timm_pretrained_model_name = ''

############################### Encoders #########################

# encoder = 'ResEncoder'


## LaserEncoder_V2  - LaserEncoder_V1R 
if encoder in ['LaserEncoder_V2', 'LaserEncoder_V1']: # Conv 3D

    if dataset == '200fps':
        encoders = [(35, (2, 3, 3), (1, 1, 1)), (128, (2, 5, 5), (2, 1, 1)), (512, (2, 3, 10), (2, 1, 1))]
        
    elif dataset == '100fps':
        encoders = [[(35, (2, 3, 3), (1, 1, 1)), (128, (2, 5, 5), (1, 1, 1)), (512, (2, 3, 10), (2, 1, 1))],
                    [(35, (2, 3, 3), (1, 1, 1)), (128, (1, 5, 5), (1, 1, 1)), (512, (2, 3, 10), (2, 1, 1))]]
                
    elif dataset == '50fps':
        encoders = [[(35, (2, 3, 3), (1, 1, 1)), (128, (2, 5, 5), (1, 1, 1)), (512, (2, 3, 10), (1, 1, 1))],
                    [(35, (2, 3, 3), (1, 1, 1)), (128, (1, 5, 5), (1, 1, 1)), (512, (1, 3, 10), (1, 1, 1))]]
        
        
    conv_2plus1d =        'True'
    bn =                  'True'
    avg_pool            = 'True'


# LaserEncoder_V2Conv2D encoder = [[output_dim, kernel_size, stride, padding],..] = layer_desc
elif encoder == 'LaserEncoder_V2Conv2D': # Conv 2D
    if dataset == '200fps':
        encoders = [(32, (1, 3), (1, 1), (0, 1)), (128, (2, 3), (2, 1), (0, 1)), (512, (2, 3), (2, 1), (0, 1))]
        
    elif dataset == '100fps':
        encoders = [[(32, (1, 3), (1, 1), (0, 1)), (128, (2, 3), (1, 1), (0, 1)), (512, (2, 3), (2, 1), (0, 1))],
                    [(32, (1, 3), (1, 1), (0, 1)), (128, (1, 3), (1, 1), (0, 1)), (512, (2, 3), (2, 1), (0, 1))]]
        
    elif dataset == '50fps':
        encoders = [[(32, (1, 3), (1, 1), (0, 1)), (128, (2, 3), (1, 1), (0, 1)), (512, (2, 3), (1, 1), (0, 1))],
                    [(32, (1, 3), (1, 1), (0, 1)), (128, (1, 3), (1, 1), (0, 1)), (512, (1, 3), (1, 1), (0, 1))]]
    
    conv_2plus1d =        'False'
    bn =                  'True'
    avg_pool            = 'True'

# LaserEncoder_V2Conv1D encoder = [[output_dim, kernel_size, stride, padding],..] = layer_desc
elif encoder == 'LaserEncoder_V2Conv1D':
    encoders = [[(1024, 1, 1, 0), (2048, 4, 4, 0)]]
    bn =                  'True'
    
###############################



feature_grad_mult =  '1.0' # or 1.0
if feature_grad_mult!= '1.0':
    print([f'{feature_grad_mult=}\n']*10)
seed = 13 # None | number #  just create another run from scratch, it does not really change any seed
H_W = '"+model.modalities.laser.input_HW=\\"(10,16)\\""'



if fp16 == 'True' and bf16 == 'True':
    raise Exception("Can not use fp16 and bf16 at the same time!")

# if normalize == 'True' and bn == 'True':
#     raise Exception("Can not use normalize and bn at the same time!")

node_list =  '' ##SBATCH --nodelist=gpu[12,13,16,17,18,19]'
#node_list = 'SBATCH --nodelist=gpu0'
channels = tuple(list(range(9)))

    
num_dims = len(channels)

experiment_base_folder = 'd2v_bust'
base_dir = f'/mnt/ML/TrainResults/katya.ivantsiv/{experiment_base_folder}'
scpt_dir = '3D'

# experiment_base_folder = 'bust'
# base_dir = f'/mnt/ML/ModelsTrainResults/katya.ivantsiv/{experiment_base_folder}'
# scpt_dir = 'encoders_search'

'''
/home/katya.ivantsiv/slurm/run_scripts/d2v_V2_v1/3D/pretrain_name_pt.sbatch
/home/katya.ivantsiv/slurm/run_scripts/d2v_V2_v1/3D/pretrain_name_finetune_name.sbatch
/mnt/ML/TrainResults/katya.ivantsiv/d2v_V2_v1/3D/pretrain_name/finetune_name/checkpoints/checkpoint_last.pt
/home/katya.ivantsiv/slurm_logs/d2v_V2_v1/pretrain_name_pt.txt
/home/katya.ivantsiv/slurm_logs/d2v_V2_v1/pretrain_name_finetune_name.txt

'''
model_save_path_base = '/mnt/ML/TrainResults/katya.ivantsiv'
# model_save_path_base = '/mnt/ML/ModelsTrainResults/katya.ivantsiv'
if div_clip != 32:
    print("You sure we don't want to use div_clip ????!!!")
# assert os.path.exists(f'/home/katya.ivantsiv/slurm_logs/{experiment_base_folder}/{scpt_dir}'), f'log folder does not exsits such  /home/katya.ivantsiv/slurm_logs/{experiment_base_folder}/{scpt_dir} '

run_validation = False
os.chdir(f"{base_dir}/{scpt_dir}")
for encoder_spec in encoders:
 
    chan_txt = channels_to_name(channels)
    
    text_constuct = []
    text_constuct.append(f'_{dataset}' if dataset else '')
    text_constuct.append(f'_large_no_nrm')
    if encoder in ['LaserEncoder_V2','LaserEncoder_V1', 'LaserEncoder_V2Conv2D', 'LaserEncoder_V2Conv1D','ImageEncoder', 'ResEncoder']:
        text_constuct.append(f'_bn' if eval(bn) else '')

    # text_constuct.append(f'_nrm' if eval(normalize) else '')
    if encoder in ['ResEncoder']:
        encoder_name = hash_encoder(encoder_spec)
        text_constuct.append(f'_res{res_encoder_output_dim}')
        text_constuct.append(f'_act{res_encoder_activation}')
        text_constuct.append(f'_2p1d' if eval(conv_2plus1d) else '')

    if encoder in ['LaserEncoder_V2', 'LaserEncoder_V1', 'LaserEncoder_V2Conv1D', 'LaserEncoder_V2Conv2D']:
        encoder_name = hash_encoder(encoder_spec)
        text_constuct.append(f'_le{encoder.split("_")[1]}')
        text_constuct.append(f'_sc' if eval(skip_connection) else '')
        if encoder in ['LaserEncoder_V2', 'LaserEncoder_V1', 'LaserEncoder_V2Conv2D']:
            text_constuct.append(f'_avgpool' if eval(avg_pool) else '')
            if encoder in ['LaserEncoder_V2', 'LaserEncoder_V1']:
                text_constuct.append(f'_2p1d' if eval(conv_2plus1d) else '')
    
    elif encoder == 'GenericEncoder':
        encoder_name = GenericEncoder_uuid[:6]
    elif encoder == 'ImageEncoder':
        encoder_name = f'{timm_pretrained_model_name}'
        text_constuct.append(f'_dim{timm_output_dim}')
        text_constuct.append(f'_st{timm_time_strides}')
        


    text_constuct.append(f'_Pstep{max_updates_pre/1000}k')
    text_constuct.append(f'_mt{max_tokens_pre/1000}k')
    text_constuct.append(f'_uf{update_freq_pre}')
    
    text_constuct.append(f'_prelr{lr_pre}'.replace('.','p'))
    
    text_constuct.append(f'_s{seed}' if seed else '')
    text_constuct.append(f'_bf16' if eval(bf16) else '')
    text_constuct.append(f'_fp32' if (not eval(bf16) and not eval(fp16)) else '')
    text_constuct.append(f'noSDPA' if not eval(SDPA) else '')
    text_constuct.append(channels_to_name(channels))

    
    exp_name = f'{encoder_name}'
    for txt in text_constuct:
        exp_name += txt
    exp_name = exp_name.replace('.','p')
    
    print(exp_name)
    exp_name_pt = exp_name +'_pt'   
    print("==============================")
    print(f'exp_name_pt: {exp_name_pt}')
    sweep_dir = f'{model_save_path_base}/{experiment_base_folder}/{scpt_dir}/{exp_name}'
    print(f'{sweep_dir=}')
    
    pretrain_sweep_dir = f'{sweep_dir}/pretrain'
    
    pretrain_sbatch = apply_with_fail(pretrain_txt,
                                      project_path=project_path,
                                    #   experiment_base_folder=experiment_base_folder,scpt_dir=scpt_dir,
                                      num_dims=num_dims, encoder_spec=encoder_spec,
                                      bn=bn.lower(), n_gpus=n_gpus, 
                                      skip_connection = skip_connection.lower(),
                                      task_data=pretrain_task_data,
                                      encoder=encoder,
                                      config_name=pretrain_config_name,
                                      max_tokens=max_tokens_pre,
                                      max_updates=max_updates_pre,
                                      update_frequency=update_freq_pre,
                                      sweep_dir=pretrain_sweep_dir,
                                      conv_2plus1d=conv_2plus1d.lower(), 
                                      exp_name_pt=exp_name_pt,
                                      features_base_path=features_base_path,
                                      channels=channels, 
                                      node_list=node_list,
                                      H_W=H_W,
                                      avg_pool=avg_pool.lower(),
                                      timm_output_dim=timm_output_dim,
                                      timm_time_strides=timm_time_strides,
                                      timm_pretrained_model_name=timm_pretrained_model_name,
                                      GenericEncoder_uuid=GenericEncoder_uuid,
                                      res_encoder_output_dim=res_encoder_output_dim,
                                      res_encoder_activation=res_encoder_activation,
                                      train_datasets=train_datasets,
                                      bf16=bf16.lower(),
                                      fp16=fp16.lower(),
                                      SDPA=SDPA.lower(),
                                      use_h5=use_h5.lower(),
                                      lr=lr_pre,
                                      partition=partition,
                                      nice=nice,
                                      div_clip=div_clip,
                                      qos=qos,
                                      SR100_support=SR100_support,
    )

    # print(pretrain_sbatch)
    pretrain_job = f'{base_dir}/{scpt_dir}/{exp_name}_pt.sbatch'
    #'''
    with open(pretrain_job,'wt') as f:
        f.write(pretrain_sbatch)
    print(f'{pretrain_job=}')


    # pretrain_output = subprocess.check_output(['sbatch',pretrain_job])
    
    
    # print(f'pretrain_output: {pretrain_output}')
    
    # jobid = pretrain_output.decode('ascii').strip().split(' ')[-1]
    
    # print(f'training job id: {jobid}')
    

    pretrained_path=f'{pretrain_sweep_dir}/0/checkpoints/checkpoint_last.pt'
    print(f'pretrained path: {pretrained_path}')

    text_constuct = ['']
    text_constuct.append(f'_fnlr{lr_fine}'.replace('.','p') if lr_fine != 0.0001 else '')
    text_constuct.append(f'_Fstep{max_updates_fine/1000}k')
    text_constuct.append(f'_mt{max_tokens_fine/1000}k')
    text_constuct.append(f'_uf{update_freq_fine}')
    
    text_constuct.append(f'_fgm{feature_grad_mult}' if float(feature_grad_mult)!=0 else '')
    text_constuct.append(f'_hbrt' if eval(hubert_soft) else '')
    text_constuct.append(f'_hbrtasr' if eval(hubert_asr) else '')
    text_constuct.append(f'noSDPA' if not eval(SDPA) else '')
    
    text_constuct.append(f'_exblock{extra_trans_blocks_depth}' if extra_trans_blocks_depth else '')
    text_constuct.append(f'_choice{use_choice_sentences}' if use_choice_sentences else '')
    text_constuct.append(f'_lip' if eval(lip_embds) else '')
    text_constuct.append(f'_liptreshold{lip_reading_treshold}' if lip_reading_treshold else '')

    text_constuct.append(f'_mask_prob{mask_prob_fine}'.replace('.','p') if mask_prob_fine != 0.05 else '')


    
    ft_features_txt = ''.join(text_constuct)
    exp_name_ft = exp_name +'_ft' + ft_features_txt 
    exp_name_ft=exp_name_ft.replace('.','p')
    finetune_sweep_dir = f'{sweep_dir}/finetune{ft_features_txt}'
    
    
    finetune_sbatch = apply_with_fail(finetune_txt,
                                      project_path=project_path,
                                      max_tokens=max_tokens_fine,
                                      max_updates=max_updates_fine,
                                      update_frequency=update_freq_fine,
                                      n_gpus=n_gpus, task_data=finetune_task_data, config_name=finetune_config_name,
                                      sweep_dir=finetune_sweep_dir,
                                      pretrained_path=pretrained_path,
                                      feature_grad_mult=feature_grad_mult,
                                      features_base_path=features_base_path,
                                      channels=channels,
                                      node_list=node_list,
                                      hubert_soft = hubert_soft.lower(),
                                      hubert_asr = hubert_asr.lower(),
                                      lr=lr_fine,
                                      bf16=bf16.lower(),
                                      fp16=fp16.lower(),
                                      SDPA=SDPA.lower(),
                                      extra_trans_blocks_depth=extra_trans_blocks_depth,
                                      use_choice_sentences=use_choice_sentences,
                                      lip_embds=lip_embds.lower(),
                                      lip_reading_path=lip_reading_path,
                                      lip_reading_treshold=lip_reading_treshold,
                                      use_h5=use_h5.lower(),
                                      mask_prob=mask_prob_fine,
                                      exp_name_ft=exp_name_ft,
                                      validation_datasets=validation_datasets,
                                      train_datasets=train_datasets,
                                      partition=partition,
                                      div_clip=div_clip,
                                      nice=nice,
                                      qos=qos,
                                     )
                                  # add_raw_image=add_raw_image)
                                  
    # print(finetune_sbatch)
    # '''
    # wdsc
    fintune_job = f'{base_dir}/{scpt_dir}/{exp_name_ft}.sbatch'
    with open(fintune_job,'wt') as f:
        f.write(finetune_sbatch)
    print(f'{fintune_job=}')
    # dfsdfsfd
    # finetune_output = subprocess.check_output(['sbatch',f'--dependency=afterok:{jobid}', fintune_job])
    finetune_output = subprocess.check_output(['sbatch', fintune_job])
    jobid_ft = finetune_output.decode('ascii').strip().split(' ')[-1]
    print(f'finetune job id: {jobid_ft}')
    print(finetune_output)
    #'''
    wefwfsfdsfsdf
    

[(32, (1, 3), (1, 1), (0, 1)), (128, (2, 3), (1, 1), (0, 1)), (512, (2, 3), (2, 1), (0, 1))] 899d8e5f
899d8e5f_100fps_large_no_nrm_bn_leV2Conv2D_sc_avgpool_Pstep150p0k_mt7p6k_uf3_prelr0p0004_s13noSDPA_0to8
exp_name_pt: 899d8e5f_100fps_large_no_nrm_bn_leV2Conv2D_sc_avgpool_Pstep150p0k_mt7p6k_uf3_prelr0p0004_s13noSDPA_0to8_pt
sweep_dir='/mnt/ML/TrainResults/katya.ivantsiv/d2v_bust/3D/899d8e5f_100fps_large_no_nrm_bn_leV2Conv2D_sc_avgpool_Pstep150p0k_mt7p6k_uf3_prelr0p0004_s13noSDPA_0to8'
pretrain_job='/mnt/ML/TrainResults/katya.ivantsiv/d2v_bust/3D/899d8e5f_100fps_large_no_nrm_bn_leV2Conv2D_sc_avgpool_Pstep150p0k_mt7p6k_uf3_prelr0p0004_s13noSDPA_0to8_pt.sbatch'
pretrained path: /mnt/ML/TrainResults/katya.ivantsiv/d2v_bust/3D/899d8e5f_100fps_large_no_nrm_bn_leV2Conv2D_sc_avgpool_Pstep150p0k_mt7p6k_uf3_prelr0p0004_s13noSDPA_0to8/pretrain/0/checkpoints/checkpoint_last.pt
fintune_job='/mnt/ML/TrainResults/katya.ivantsiv/d2v_bust/3D/899d8e5f_100fps_large_no_nrm_bn_leV2Conv2D_sc_avgpool_Pstep15

NameError: name 'wefwfsfdsfsdf' is not defined

In [ ]:
mkdir -p /home/katya.ivantsiv/slurm_logs/d2v_bust/3D 

OSError: "/bin/zsh" shell not found